In [23]:
import pandas as pd
import numpy as np
import sqlite3
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

In [24]:
with sqlite3.connect("../data/pancreas_proteomics.db") as conn:
    df = pd.read_sql("SELECT * FROM imputed_KNN", conn)
    condition = pd.read_sql("SELECT [Sample ID], Condition FROM expression GROUP BY [Sample ID]", conn)
    df_ann = pd.read_sql("SELECT [Protein ID], [Annotated Matrisome Category] FROM annotations", conn)

df_pivot = df.pivot(index='Protein ID', columns='Sample ID', values='Intensity')
X = df_pivot.T

In [25]:
y = X.index.map(condition.set_index('Sample ID')['Condition'])
y_binary = (y == 'T1D').astype(int)
y_binary

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [26]:
rf = RandomForestClassifier(n_estimators=500, random_state=42, max_depth=5)
loo = LeaveOneOut()

In [27]:
predictions = []
for train, test in loo.split(X):
    X_train, X_test = X.iloc[train], X.iloc[test]
    y_train, y_test = y_binary[train], y_binary[test]
    rf.fit(X_train, y_train)
    predictions.append(rf.predict(X_test)[0])

In [28]:
accuracy = accuracy_score(y_binary, predictions)
accuracy

0.8888888888888888

In [29]:
rf.fit(X, y_binary)
importances = pd.DataFrame({'Protein ID': X.columns,'Importance': rf.feature_importances_}).sort_values(by='Importance', ascending=False).head(15)


In [30]:
importances = importances.merge(df_ann, on='Protein ID', how='left')
importances['Label'] = importances['Annotated Matrisome Category'].fillna(importances['Protein ID'])
importances

,Protein ID,Importance,Annotated Matrisome Category,Label
0,Q86WB0,0.010,Non-matrisome,Non-matrisome
1,P52566,0.010,Non-matrisome,Non-matrisome
2,Q9NVT9,0.008,Non-matrisome,Non-matrisome
3,Q92785,0.008,Non-matrisome,Non-matrisome
4,Q8WUF5,0.008,Non-matrisome,Non-matrisome
5,P49792,0.008,Non-matrisome,Non-matrisome
6,P51692,0.008,Non-matrisome,Non-matrisome
7,Q5VZF2,0.006,Non-matrisome,Non-matrisome
8,A0A2R8Y619,0.006,Non-matrisome,Non-matrisome
9,Q9NRR5,0.006,Non-matrisome,Non-matrisome
